# Hierarchical Clustering
Accompanies the notes on **Distance Matrices and Hierarchical Clustering**.

## 1. Merges by hand: four points, four linkages

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist, squareform

pts = np.array([[1, 1], [2, 1], [5, 4], [6, 5]])          # A, B, C, D
print(squareform(pdist(pts)).round(3))
for method in ["single", "complete", "average", "ward"]:
    Z = linkage(pts, method=method)
    print(f"{method:9s} merge heights: {Z[:, 2].round(3)}")

## 2. Why units matter

In [ ]:
import pandas as pd
cust = pd.DataFrame({"age": [25, 26, 60], "income": [50000, 90000, 51000]}, index=["P", "Q", "R"])
print(squareform(pdist(cust)).round(1))
z = (cust - cust.mean()) / cust.std(ddof=0)
print(squareform(pdist(z)).round(2))

## 3. The linkages side by side: moons and blobs

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
from scipy.cluster.hierarchy import fcluster

moons, _ = make_moons(n_samples=200, noise=0.06, random_state=0)
blobs, _ = make_blobs(n_samples=200, centers=[[0, 0], [3, 0], [1.5, 2.5]], cluster_std=[0.4, 0.4, 0.4], random_state=0)
blobs = np.vstack([blobs, [[4.6, 1.6]]])                  # one stray point
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for row, data in enumerate([moons, blobs]):
    k = 2 if row == 0 else 3
    for col, method in enumerate(["single", "complete", "average", "ward"]):
        labels = fcluster(linkage(data, method=method), t=k, criterion="maxclust")
        axes[row, col].scatter(data[:, 0], data[:, 1], c=labels, s=10, cmap="viridis")
        axes[row, col].set_title(f"{method}, {k} clusters"); axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        print(f"{'moons' if row == 0 else 'blobs'} {method:9s} cluster sizes: {np.bincount(labels)[1:]}")
plt.tight_layout(); plt.show()

## 4. Clustering 300 bank customers

In [ ]:
bd = pd.read_csv("../../data/bd_train.csv")
cols = ["Average.Credit.Card.Transaction", "Balance.Transfer", "Term.Deposit", "Life.Insurance",
        "Medical.Insurance", "Average.A.C.Balance", "Personal.Loan", "Investment.in.Mutual.Fund",
        "Investment.Tax.Saving.Bond", "Home.Loan", "Online.Purchase.Amount", "Investment.in.Commudity",
        "Investment.in.Equity", "Investment.in.Derivative", "Portfolio.Balance"]
sample = bd.sample(300, random_state=0)
x = sample[cols]
xs = (x - x.mean()) / x.std(ddof=0)
from scipy.cluster.hierarchy import dendrogram, cophenet
for method in ["single", "complete", "average", "ward"]:
    Z = linkage(xs, method=method)
    sizes = np.bincount(fcluster(Z, t=4, criterion="maxclust"))[1:]
    print(f"{method:9s} cophenetic r = {cophenet(Z, pdist(xs))[0]:.3f}   4-cluster sizes: {sorted(sizes.tolist(), reverse=True)}")
Z = linkage(xs, method="ward")
plt.figure(figsize=(10, 4))
dendrogram(Z, truncate_mode="lastp", p=30, no_labels=True)
plt.ylabel("merge height (Ward)"); plt.title("Ward dendrogram, 300 bank customers (last 30 merges shown)")
plt.tight_layout(); plt.show()
print("last merge heights:", Z[-6:, 2].round(1))

## 5. What are the clusters? Profiles

In [ ]:
sample["cluster"] = fcluster(Z, t=4, criterion="maxclust")
profile = sample.groupby("cluster")[cols[:6] + ["Portfolio.Balance"]].median().round(0)
profile["size"] = sample["cluster"].value_counts().sort_index()
profile["Revenue.Grid=1 (%)"] = (sample.groupby("cluster")["Revenue.Grid"].apply(lambda s: (s == 1).mean()) * 100).round(1)
print(profile.T)

## Try it yourself
1. Rerun section 4 without standardising (`xs = x`). Which columns decide the clusters?
2. Cut the Ward tree at 2 and at 6 clusters (`t=2`, `t=6`) and compare the profiles.